# Connect resources — Teamwork Cloud (TWC)

This notebook shows how to **connect** a resource that lives in **Dassault Teamwork Cloud** without copying the project into Istari. You upload a small **pointer file** on the platform; that file is only metadata — a link (`mdel://…`) to the real source, which **stays in TWC**. When you later run `@istari:twc_extract`, the job uses that link (and your credentials) to read from Teamwork Cloud on the agent.

The pointer is JSON with one field, `identifier`, holding the full project link from the TWC web app. The platform expects the filename extension `.istari_teamwork_cloud_metadata_mdzip`. Because `add_model` and `add_function_auth_secret` require a filesystem path, the recipe builds those payloads in memory and uploads them via short-lived temp files.

Uses the official v2 **`istari_digital_client.Client`** (same pattern as [`chaining_jobs_no_helper.ipynb`](../chaining_jobs_no_helper.ipynb)). Reads [`samples/.env`](../.env) for platform and TWC credentials.

### What the steps do

1. Upload the pointer file — register the **connected** link on Istari (`add_model`).
2. Register TWC login as a function-auth secret for the extract job.
3. Submit `@istari:twc_extract`, poll, list output artifacts, and download selected JSON outputs.

### Prerequisites

- **`istari-digital-client`** and **`python-dotenv`** (from cookbook root: `uv sync --group dev`).
- Access to **`@istari:twc_extract`** and a Cameo-capable agent pool (`dassault_cameo` on Windows Server 2022).
- In `samples/.env`: `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`, `TWC_USERNAME`, `TWC_PASSWORD` (do not commit `.env`).
- A **project link** from Teamwork Cloud — set `PROJECT_LINK` in **Prep**.

### Install kernel (optional)

From the cookbook repository root:

```bash
uv sync --group dev
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Select **Python (istari-client-cookbook)** in the kernel picker.

### Running cells individually

Run **Setup → Connect**, then **Prep**, then §1–§5 in order. **Connect** matches [`using-resources.ipynb`](./using-resources.ipynb) (`Configuration` + `Client` only). **Prep** holds `PROJECT_LINK`, display names, and `twc_pointer_payload`.

## Setup

**Connect** loads credentials and constructs `Client`. **Prep** sets TWC constants, env asserts, `twc_pointer_payload`, and `ui_base` — edit for your project before the demo cells.

### Connect

Load [`samples/.env`](../.env), build `Configuration`, and create `Client` — the same three-step pattern as other cookbook recipes.

In [ ]:
import os
from importlib.metadata import version as pkg_version
from pathlib import Path

import dotenv
from istari_digital_client import Client, Configuration

EXPECTED_CLIENT_VERSION = "10.10.0"

_cwd = Path.cwd()
SAMPLES_DIR = _cwd.parent if (_cwd.parent / ".env").exists() else _cwd / "samples"

dotenv.load_dotenv(SAMPLES_DIR / ".env", override=True)
registry_url = os.environ.get("ISTARI_REGISTRY_URL")
token = os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
if not registry_url or not token:
    raise RuntimeError("Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env")

installed = pkg_version("istari-digital-client")
assert installed == EXPECTED_CLIENT_VERSION, (
    f"Expected istari-digital-client=={EXPECTED_CLIENT_VERSION}, got {installed}"
)

client = Client(Configuration(registry_url=registry_url, registry_auth_token=token))

print(f"istari-digital-client {EXPECTED_CLIENT_VERSION}")
print(f"Registry: {registry_url}")

### Prep

TWC project link, display names for the connected resource, pointer file extension, and asserts for `TWC_USERNAME` / `TWC_PASSWORD` from `.env`. Change these for your project; they are not required to connect to the platform.

In [ ]:
import json
import os
import re
import tempfile
from pathlib import Path
from time import sleep

from istari_digital_client import FunctionAuthType, JobStatusName, NewSource

POINTER_SUFFIX = ".istari_teamwork_cloud_metadata_mdzip"

# Copy from TWC web app — stored verbatim in pointer JSON as "identifier".
PROJECT_LINK = (
    "mdel://ANY?projectID=twcloud:/cf16d07a-9a9f-43dd-8ad9-bee73d9e2581/"
    "9903a418-e792-4a69-9add-dc01fe2f57de&serverType=esiserver&serverName=10.30.102.103"
    "&projectName=Istari_UAVOne(notional)&elementName=ANY_NAME"
)

POINTER_DISPLAY_NAME = "Istari_UAVOne(notional)"
POINTER_EXTERNAL_ID = "Istari_UAVOne_notional"

TWC_USERNAME = os.environ.get("TWC_USERNAME")
TWC_PASSWORD = os.environ.get("TWC_PASSWORD")
assert TWC_USERNAME, "Set TWC_USERNAME in samples/.env"
assert TWC_PASSWORD, "Set TWC_PASSWORD in samples/.env"

_match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", registry_url)
ui_base = registry_url.rstrip("/") if not _match else f"{_match.group(1)}{_match.group(2)}"

def twc_pointer_payload(project_link: str) -> dict:
    """TWC pointer file body: one identifier field with the full mdel:// link."""
    return {"identifier": project_link}


print(f"TWC user: {TWC_USERNAME}")
print(f"Project link: {PROJECT_LINK[:60]}…")

## 1 · Upload the TWC pointer file

Register the connected link on the platform: JSON `{"identifier": "<mdel:// project link>"}` with extension `.istari_teamwork_cloud_metadata_mdzip`. The SysML project remains in Teamwork Cloud; Istari stores only this pointer. Upload uses `add_model` via a short-lived temp file (the SDK has no in-memory upload).

In [ ]:
pointer_payload = twc_pointer_payload(PROJECT_LINK)
print(f"Pointer file content ({POINTER_SUFFIX}):")
print(json.dumps(pointer_payload, indent=2))

_pointer_tmp = tempfile.NamedTemporaryFile(
    mode="w", encoding="utf-8", suffix=POINTER_SUFFIX, delete=False
)
json.dump(pointer_payload, _pointer_tmp, indent=2)
_pointer_tmp.write("\n")
_pointer_tmp.close()
_pointer_path = Path(_pointer_tmp.name)

try:
    print(f"Uploading pointer {_pointer_path.name}...")
    model = client.add_model(
        path=_pointer_path,
        external_identifier=POINTER_EXTERNAL_ID,
        display_name=POINTER_DISPLAY_NAME,
        description="TWC connected resource (mdel project link)",
    )
finally:
    _pointer_path.unlink(missing_ok=True)

print(f"Connected resource id: {model.id}")
print(f"UI: {ui_base}/files/{model.id}")

## 2 · Register TWC auth

Build `{"username", "password"}` from `TWC_USERNAME` / `TWC_PASSWORD`, upload as a **function auth secret** (Basic) via a temp file, then reference it on the job with `NewSource` and relationship `twc_auth_login`.

In [ ]:
_secret_tmp = tempfile.NamedTemporaryFile(
    mode="w", encoding="utf-8", suffix=".json", delete=False
)
json.dump({"username": TWC_USERNAME, "password": TWC_PASSWORD}, _secret_tmp, indent=2)
_secret_tmp.write("\n")
_secret_tmp.close()
_secret_path = Path(_secret_tmp.name)

try:
    secret = client.add_function_auth_secret(
        path=_secret_path,
        function_auth_type=FunctionAuthType.BASIC,
    )
finally:
    _secret_path.unlink(missing_ok=True)

auth_source = NewSource(
    revision_id=secret.revision.id,
    relationship_identifier="twc_auth_login",
)
print(f"TWC auth secret revision: {auth_source.revision_id}")

## 3 · Run `@istari:twc_extract`

Submit a job against the pointer resource from §1, with the auth source from §2 and Cameo tool metadata. The function reads from Teamwork Cloud; it does not upload or register the project itself. Optionally set `parameters` (e.g. `root_element_id`) for a subtree — `elementName=ANY_NAME` in the project link is not passed automatically.

In [ ]:
job = client.add_job(
    model_id=model.id,
    function="@istari:twc_extract",
    tool_name="dassault_cameo",
    tool_version="2022x Refresh2",
    operating_system="Windows Server 2022",
    sources=[auth_source],
)
print(f"Job {job.id} submitted")
print(f"UI: {ui_base}/jobs/{job.id}")

## 4 · Poll until the job finishes

Poll every five seconds until status is **Completed**, **Failed**, or **Canceled**.

In [ ]:
elapsed = 0
poll_interval = 5

while True:
    job = client.get_job(job.id)
    status = job.status.name
    print(f"{elapsed}s: {status.value}")
    if status in (
        JobStatusName.COMPLETED,
        JobStatusName.FAILED,
        JobStatusName.CANCELED,
    ):
        break
    sleep(poll_interval)
    elapsed += poll_interval

if job.status.name != JobStatusName.COMPLETED:
    msg = job.status.message or ""
    raise RuntimeError(f"Job {job.id} ended with {job.status.name!s}. {msg}")

## 5 · List artifacts

After a successful extract, output artifacts are attached to the pointer resource you uploaded in §1.

In [ ]:
model = client.get_model(model.id)
print("Artifacts:")
for artifact in model.artifacts or []:
    print(f"  {artifact.id}  {artifact.name}  {artifact.external_identifier}")

## 6 · Download artifacts

Save extract outputs locally with `read_contents` (same pattern as [`chaining_jobs_no_helper.ipynb`](../chaining_jobs_no_helper.ipynb) §5).

In [ ]:
DOWNLOAD_DIR = Path.cwd()
ARTIFACTS_TO_DOWNLOAD = ("sysml_elements.json", "requirements.json")


def download_artifact(model_id: str, filename: str, *, out_dir: Path = DOWNLOAD_DIR) -> Path:
    """Download a named artifact from the model to out_dir."""
    model = client.get_model(model_id)
    match = next((a for a in model.artifacts or [] if a.name == filename), None)
    if match is None:
        names = [a.name for a in model.artifacts or []]
        raise RuntimeError(f"{filename} not on model — found: {names}")

    artifact = client.get_artifact(match.id)
    revision = artifact.file.revision if artifact.file else None
    if revision is None or revision.content_token is None:
        raise RuntimeError(f"{filename} has no readable revision")

    raw = client.read_contents(token=revision.content_token)
    dest = out_dir / filename
    dest.write_bytes(raw)
    return dest


for name in ARTIFACTS_TO_DOWNLOAD:
    path = download_artifact(model.id, name)
    print(f"Wrote {path.resolve()} ({path.stat().st_size} bytes)")
